In [1]:
!pip install -q -U transformers datasets accelerate
import torch; print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 87.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 82.0 MB/s eta 0:00:00:00:01
True Tesla T4


In [1]:
import re
from datasets import load_dataset

LABELS = ['OM','SD','SA','KW','QA','LB','JO','SY','IQ','MA',
          'EG','PL','YE','BH','DZ','AE','TN','LY']

def clean(text):
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"#", " ", text)
    text = re.sub(r"[\u064B-\u0652]", "", text)
    return re.sub(r"\s+", " ", text).strip()

ds = load_dataset("Abdelrahman-Rezk/Arabic_Dialect_Identification")
ds = ds.map(lambda x: {"text": clean(x["text"])})
N_TRAIN = 100_000
train = ds["train"].shuffle(seed=42)
val, test = ds["validation"], ds["test"]

README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/47.7M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/1.00M [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/9164 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8981 [00:00<?, ? examples/s]

Map:   0%|          | 0/440052 [00:00<?, ? examples/s]

Map:   0%|          | 0/9164 [00:00<?, ? examples/s]

Map:   0%|          | 0/8981 [00:00<?, ? examples/s]

In [2]:
from transformers import AutoTokenizer
MODEL = "UBC-NLP/MARBERTv2"
tok = AutoTokenizer.from_pretrained(MODEL)

def tokenize(batch):
    return tok(batch["text"], truncation=True, max_length=64)

train_t = train.map(tokenize, batched=True, remove_columns=["id","text"])
val_t   = val.map(tokenize, batched=True, remove_columns=["id","text"])
test_t  = test.map(tokenize, batched=True, remove_columns=["id","text"])

config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/440052 [00:00<?, ? examples/s]

Map:   0%|          | 0/9164 [00:00<?, ? examples/s]

Map:   0%|          | 0/8981 [00:00<?, ? examples/s]

In [3]:
import numpy as np
from transformers import (AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding)
from sklearn.metrics import f1_score

model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=18)

def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    return {"macro_f1": f1_score(p.label_ids, preds, average="macro")}

args = TrainingArguments(
    output_dir="/kaggle/working/marbert-dialect",
    learning_rate=2e-5,
    warmup_steps=400,
    weight_decay=0.01,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    num_train_epochs=2,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=200,
    report_to="none",
)
trainer = Trainer(model=model, args=args, train_dataset=train_t, eval_dataset=val_t,
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics)
trainer.train()

pytorch_model.bin:   0%|          | 0.00/654M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/654M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect ide

Epoch,Training Loss,Validation Loss,Macro F1
1,2.480655,2.375903,0.603911
2,2.075189,2.261990,0.623521


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=6876, training_loss=2.546328200095611, metrics={'train_runtime': 5788.4303, 'train_samples_per_second': 152.045, 'train_steps_per_second': 1.188, 'total_flos': 2.8550137471501824e+16, 'train_loss': 2.546328200095611, 'epoch': 2.0})

In [7]:
from sklearn.metrics import classification_report
out = trainer.predict(test_t)
preds = out.predictions.argmax(-1)
print("Macro-F1:", round(f1_score(test_t["label"], preds, average="macro"), 4))
print(classification_report(test_t["label"], preds, target_names=LABELS))

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Macro-F1: 0.6144
              precision    recall  f1-score   support

          OM       0.57      0.48      0.52       375
          SD       0.83      0.64      0.72       283
          SA       0.51      0.57      0.54       526
          KW       0.62      0.69      0.65       825
          QA       0.55      0.54      0.55       609
          LB       0.71      0.76      0.73       541
          JO       0.54      0.44      0.49       547
          SY       0.59      0.44      0.51       318
          IQ       0.72      0.65      0.68       304
          MA       0.81      0.64      0.71       226
          EG       0.80      0.92      0.85      1130
          PL       0.60      0.66      0.63       857
          YE       0.42      0.39      0.41       195
          BH       0.49      0.43      0.46       515
          DZ       0.68      0.64      0.66       317
          AE       0.48      0.53      0.51       516
          TN       0.79      0.57      0.66       181
          

In [4]:
# حفظ نهائي + اختبار إنه بيتحمّل صح
SAVE = "/kaggle/working/marbert-dialect-final"
trainer.save_model(SAVE)
tok.save_pretrained(SAVE)

from transformers import pipeline
reloaded = AutoModelForSequenceClassification.from_pretrained(SAVE)
clf = pipeline("text-classification", model=reloaded, tokenizer=tok, device=0)

tests = ["ايه الاخبار يا معلم عامل ايه",
         "شنو الاخبار يا زول كيفك",
         "وش السالفة يا رجال",
         "شو الاخبار كيفك"]
for t in tests:
    r = clf(t)[0]
    print(LABELS[int(r["label"].split("_")[1])], round(r["score"], 2), "|", t)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

EG 0.79 | ايه الاخبار يا معلم عامل ايه
SD 0.99 | شنو الاخبار يا زول كيفك
SA 0.76 | وش السالفة يا رجال
LB 0.63 | شو الاخبار كيفك


In [5]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))

REPO = "zoro6u/marbert-arabic-dialect-id"   # غيّر zoro6u لو الـ username على HF مختلف
reloaded.push_to_hub(REPO)
tok.push_to_hub(REPO)
print("https://huggingface.co/" + REPO)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

https://huggingface.co/zoro6u/marbert-arabic-dialect-id


In [6]:
reloaded.config.id2label = {i: l for i, l in enumerate(LABELS)}
reloaded.config.label2id = {l: i for i, l in enumerate(LABELS)}
reloaded.push_to_hub(REPO)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/zoro6u/marbert-arabic-dialect-id/commit/44457c42aa33d5bc6aeef35c0d50afb6d1c5a708', commit_message='Upload BertForSequenceClassification', commit_description='', oid='44457c42aa33d5bc6aeef35c0d50afb6d1c5a708', pr_url=None, repo_url=RepoUrl('https://huggingface.co/zoro6u/marbert-arabic-dialect-id', endpoint='https://huggingface.co', repo_type='model', repo_id='zoro6u/marbert-arabic-dialect-id'), pr_revision=None, pr_num=None)